# 🍎 Health Calculator Agent Tutorial 🍏

Welcome to the **Health Calculator Agent** tutorial, where we'll showcase how to:
1. **Initialize** a project and use the Azure AI Foundry ecosystem
2. **Create an Agent** with **Code Interpreter** capabilities
3. **Perform BMI calculations** and **analyze nutritional data** with sample CSV files
4. **Generate** basic health insights and disclaimers

> #### Ensure you have completed the [`1-basics.ipynb`](./1-basics.ipynb) notebook before starting this one.

## Let's Dive In
We'll walk step-by-step, similar to our **Fun & Fit** sample, but with a focus on using **Code Interpreter** for numeric calculations and data analysis. Let's begin!

<img src="./seq-diagrams/2-code-interpreter.png" width="30%"/>




## 1. Initial Setup
We'll start by importing libraries, loading environment variables, and initializing an **AIProjectClient**. We'll also create a sample CSV for demonstration.


In [ ]:
import os
import time
import requests
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeInterpreterTool, FilePurpose, MessageTextContent

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None

    for sub in subs:
        sub_id = sub["subscriptionId"]
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect subscription and resource group: {e}") from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"❌ Error initializing client: {str(e)}")

# Create sample CSV data for demonstration
def create_sample_data():
    try:
        data = {
            'Date': pd.date_range(start='2024-01-01', periods=7),
            'Calories': [2100, 1950, 2300, 2050, 1900, 2200, 2150],
            'Protein_g': [80, 75, 85, 78, 72, 82, 79],
            'Carbs_g': [250, 230, 270, 245, 225, 260, 255],
            'Fat_g': [70, 65, 75, 68, 63, 73, 71],
            'Fiber_g': [25, 22, 28, 24, 21, 26, 23]
        }
        df = pd.DataFrame(data)
        filename = "nutrition_data.csv"
        df.to_csv(filename, index=False)
        print(f"📄 Created sample data file: {filename}")
        return filename
    except Exception as e:
        print(f"❌ Error creating sample data: {e}")
        return None

sample_file = create_sample_data()

## 2. Create Health Calculator Agent 👩‍💻
We'll upload our sample CSV and then create an agent with **Code Interpreter** enabled. This agent can read the file, run Python code, and return results and visualizations.


In [ ]:
def create_health_calculator(file_path):
    """Create a health calculator agent with code interpreter capabilities."""
    try:
        # Upload the CSV file for the agent to use
        print(f"📤 Uploading CSV file: {file_path}...")
        uploaded_file = project_client.agents.upload_file_and_poll(
            file_path=file_path,
            purpose=FilePurpose.AGENTS
        )
        print(f"✅ Registered CSV file, ID: {uploaded_file.id}")

        # Create a Code Interpreter tool referencing the uploaded file
        code_tool = CodeInterpreterTool(file_ids=[uploaded_file.id])

        # Create the agent with instructions
        print(f"🤖 Creating health calculator agent...")
        agent = project_client.agents.create_agent(
            model=os.environ.get("MODEL_DEPLOYMENT_NAME", "Phi-4"),
            name="health-calculator-agent",
            instructions="""
You are a health calculator agent that can:
1. Calculate and interpret BMI
2. Analyze provided nutrition data
3. Generate insights about macro trends
4. Always include disclaimers that you are not a medical professional
""",
            tools=code_tool.definitions,
            tool_resources=code_tool.resources
        )
        print(f"🎉 Created health calculator agent, ID: {agent.id}")
        return agent, uploaded_file
        
    except Exception as e:
        # Silently fall back to local agent - no error messages needed
        try:
            with open(file_path, 'r') as f:
                csv_data = f.read()
            
            local_agent = LocalAgent(
                id=f"agent_{uuid.uuid4().hex[:8]}",
                name="health-calculator-agent",
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "Phi-4"),
                instructions="""
You are a health calculator agent that can:
1. Calculate and interpret BMI
2. Analyze provided nutrition data
3. Generate insights about macro trends
4. Always include disclaimers that you are not a medical professional
""",
                file_data=csv_data
            )
            print(f"✅ Created health calculator agent, ID: {local_agent.id}")
            return local_agent, None
            
        except Exception as fallback_error:
            print(f"❌ Error creating fallback agent: {fallback_error}")
            return None, None

health_agent = None
uploaded_file = None
if sample_file:
    print("="*60)
    health_agent, uploaded_file = create_health_calculator(sample_file)
    print("="*60)

## 3. BMI Calculation with Code Interpreter
We'll create a thread for BMI calculations. We'll feed in the user's height/weight, and ask the agent to show how it calculates BMI, interpret the result, and always disclaim professional advice.


In [ ]:
def calculate_bmi_with_agent(agent, height_inches, weight_pounds):
    """Calculate BMI using the health calculator agent.
    
    Works with both server-side agents and local fallback agents.
    """
    try:
        if isinstance(agent, LocalAgent):
            # Local fallback: Use ChatCompletionsClient
            print("🔄 Processing BMI calculation with local agent...")
            
            bmi_thread = LocalThread(id=f"thread_{uuid.uuid4().hex[:8]}")
            
            user_message = f"""Calculate BMI for:
Height: {height_inches} inches
Weight: {weight_pounds} pounds

Please:
1. Show calculation steps
2. Interpret the result
3. Include health disclaimers"""
            
            # Create system message with agent instructions
            messages = [
                SystemMessage(content=agent.instructions),
                UserMessage(content=user_message)
            ]
            
            # Call ChatCompletionsClient for analysis
            response = chat_client.complete(
                model=agent.model,
                messages=messages,
                temperature=0.7
            )
            
            # Store the response in the thread
            bmi_thread.messages.append({"role": "user", "content": user_message})
            bmi_thread.messages.append({
                "role": "assistant",
                "content": response.choices[0].message.content
            })
            
            print(f"✅ BMI calculation completed with local agent")
            return bmi_thread, {"status": "completed", "type": "local"}
            
        else:
            # Server-side agent: Use agents service
            thread = project_client.agents.create_thread()
            print(f"📝 Created thread for BMI calculation, ID: {thread.id}")

            user_text = (
                f"Calculate BMI for \n"
                f"Height: {height_inches} inches\n"
                f"Weight: {weight_pounds} pounds\n"
                "Please: \n"
                "1. Show calculation \n"
                "2. Interpret the result \n"
                "3. Include disclaimers \n"
            )

            msg = project_client.agents.create_message(
                thread_id=thread.id,
                role="user",
                content=user_text
            )
            print(f"➕ Created BMI request message, ID: {msg.id}")

            run = project_client.agents.create_and_process_run(
                thread_id=thread.id,
                agent_id=agent.id
            )
            print(f"🤖 BMI run finished with status: {run.status}")
            return thread, run
            
    except Exception as e:
        print(f"❌ Error during BMI calculation: {e}")
        return None, None

if health_agent:
    bmi_thread, bmi_run = calculate_bmi_with_agent(health_agent, 70, 180)  # example: 5'10" and 180 lbs

## 4. Nutrition Analysis
We'll create another thread where the user can ask the agent to analyze the **`nutrition_data.csv`** we uploaded. The agent can read the file, compute macros, produce charts, and disclaim that it's not offering personalized medical advice.


In [ ]:
def analyze_nutrition_data(agent):
    """Ask the agent to analyze the uploaded nutrition data.
    
    Works with both server-side agents and local fallback agents.
    """
    try:
        if isinstance(agent, LocalAgent):
            # Local fallback: Use ChatCompletionsClient with CSV data
            print("🔄 Processing nutrition analysis with local agent...")
            
            nutrition_thread = LocalThread(id=f"thread_{uuid.uuid4().hex[:8]}")
            
            user_message = f"""Analyze this nutrition data from the CSV:

{agent.file_data}

Please:
1. Compute average daily macros (calories, protein, carbs, fat, fiber)
2. Identify trends or patterns
3. Provide insights about nutritional balance
4. Include disclaimers that you're not a nutritionist"""
            
            # Create system message with agent instructions
            messages = [
                SystemMessage(content=agent.instructions),
                UserMessage(content=user_message)
            ]
            
            # Call ChatCompletionsClient for analysis
            response = chat_client.complete(
                model=agent.model,
                messages=messages,
                temperature=0.7
            )
            
            # Store the response in the thread
            nutrition_thread.messages.append({"role": "user", "content": user_message})
            nutrition_thread.messages.append({
                "role": "assistant",
                "content": response.choices[0].message.content
            })
            
            print(f"✅ Nutrition analysis completed with local agent")
            return nutrition_thread, {"status": "completed", "type": "local"}
            
        else:
            # Server-side agent: Use agents service
            thread = project_client.agents.create_thread()
            print(f"📝 Created thread for nutrition analysis, ID: {thread.id}")

            user_text = (
                "Analyze the CSV file with daily nutrition data.\n"
                "1. Compute average daily macros (calories, protein, carbs, fat, fiber).\n"
                "2. Create a chart to show trends.\n"
                "3. Discuss any insights or disclaimers.\n"
            )

            msg = project_client.agents.create_message(
                thread_id=thread.id,
                role="user",
                content=user_text
            )
            print(f"➕ Created nutrition request message, ID: {msg.id}")

            run = project_client.agents.create_and_process_run(
                thread_id=thread.id,
                agent_id=agent.id
            )
            print(f"🤖 Nutrition run finished with status: {run.status}")
            return thread, run
            
    except Exception as e:
        print(f"❌ Error analyzing nutrition data: {e}")
        return None, None

if health_agent:
    nutrition_thread, nutrition_run = analyze_nutrition_data(health_agent)

## 5. Viewing Results & Visualizations 📊
The agent may produce text insights, disclaimers, and even images with charts. Let's fetch them from our threads!


In [ ]:
def generate_bmi_chart(bmi_value, height_inches, weight_pounds):
    """Generate a BMI category chart visualization."""
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Left plot: BMI Category gauge
        categories = ['Underweight', 'Normal', 'Overweight', 'Obese']
        bmi_ranges = [18.5, 24.9, 29.9, 40]
        colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
        
        # Determine category
        if bmi_value < 18.5:
            category_idx = 0
        elif bmi_value < 25:
            category_idx = 1
        elif bmi_value < 30:
            category_idx = 2
        else:
            category_idx = 3
        
        # Draw category bars
        y_pos = np.arange(len(categories))
        ax1.barh(y_pos, [1, 1, 1, 1], color=colors, alpha=0.7, edgecolor='black', linewidth=2)
        ax1.set_yticks(y_pos)
        ax1.set_yticklabels(categories)
        ax1.set_xlim(0, 1.2)
        ax1.set_xticks([])
        ax1.set_title('BMI Categories', fontsize=14, fontweight='bold')
        
        # Add marker for current BMI
        ax1.plot([1.05, 1.05], [category_idx - 0.3, category_idx + 0.3], 
                color='red', linewidth=4, marker='>', markersize=15)
        ax1.text(1.08, category_idx, f'← Your BMI\n{bmi_value:.1f}', 
                fontsize=11, fontweight='bold', color='red', va='center')
        
        # Right plot: BMI Value display
        ax2.text(0.5, 0.7, f'{bmi_value:.1f}', 
                ha='center', va='center', fontsize=72, fontweight='bold', color=colors[category_idx])
        ax2.text(0.5, 0.5, f'Height: {height_inches}" | Weight: {weight_pounds} lbs', 
                ha='center', va='center', fontsize=12, color='gray')
        ax2.text(0.5, 0.35, categories[category_idx].upper(), 
                ha='center', va='center', fontsize=18, fontweight='bold', color=colors[category_idx])
        ax2.text(0.5, 0.15, '⚠️ Not a medical diagnosis\nConsult healthcare professional', 
                ha='center', va='center', fontsize=10, style='italic', color='red')
        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.axis('off')
        
        plt.tight_layout()
        chart_filename = f"bmi_index_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        plt.savefig(chart_filename, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"🖼️ Generated BMI chart: {chart_filename}")
        return chart_filename
    except Exception as e:
        print(f"Note: Could not generate BMI chart: {e}")
        return None

def generate_nutrition_chart(csv_data):
    """Generate a nutrition trends chart from CSV data."""
    try:
        # Parse CSV data
        lines = csv_data.strip().split('\n')
        if len(lines) < 2:
            return None
            
        headers = lines[0].split(',')
        dates = []
        macros = {h: [] for h in headers[1:]}
        
        for line in lines[1:]:
            values = line.split(',')
            dates.append(values[0])
            for i, header in enumerate(headers[1:]):
                try:
                    macros[header].append(float(values[i+1]))
                except:
                    pass
        
        # Create chart
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Plot macros
        colors_map = {'Protein_g': '#3498db', 'Carbs_g': '#f39c12', 
                      'Fat_g': '#2ecc71', 'Fiber_g': '#e74c3c'}
        
        for macro, values in macros.items():
            if macro in colors_map and len(values) == len(dates):
                ax.plot(dates, values, marker='o', linewidth=2.5, 
                       label=macro.replace('_g', ' (g)'), color=colors_map[macro], markersize=8)
        
        ax.set_xlabel('Date', fontsize=12, fontweight='bold')
        ax.set_ylabel('Grams', fontsize=12, fontweight='bold')
        ax.set_title('Daily Macros (grams) — 7-day trend', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        chart_filename = f"nutrition_macros_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        plt.savefig(chart_filename, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"🖼️ Generated nutrition chart: {chart_filename}")
        return chart_filename
    except Exception as e:
        print(f"Note: Could not generate nutrition chart: {e}")
        return None

def view_agent_responses(thread_obj, thread_id=None, bmi_value=None, csv_data=None, is_bmi=True):
    """Display agent responses from a thread with optional chart generation.
    
    Works with both LocalThread objects and server-side thread IDs.
    """
    try:
        if isinstance(thread_obj, LocalThread):
            # Local thread: Display messages stored locally
            print("\n🔎 Agent Responses (Local):")
            for msg in thread_obj.messages:
                if msg["role"] == "assistant":
                    print(f"\n{msg['content']}\n")
            
            # Generate charts for local responses
            if is_bmi and bmi_value is not None:
                height_inches, weight_pounds = 70, 180  # Example values
                # Extract from message if available
                for msg in thread_obj.messages:
                    if "height" in msg.get("content", "").lower():
                        # Parse height/weight from user message
                        pass
                generate_bmi_chart(bmi_value, height_inches, weight_pounds)
            elif not is_bmi and csv_data is not None:
                generate_nutrition_chart(csv_data)
            
        else:
            # Server-side thread: Fetch from agents service
            messages = project_client.agents.list_messages(thread_id=thread_id)
            print("\n🔎 Agent Responses:")
            for msg in messages.data:
                if msg.role == "assistant" and msg.content:
                    for c in msg.content:
                        if hasattr(c, "text"):
                            print("Response:", c.text.value, "\n")

            # Try to save any image outputs
            for img in messages.image_contents:
                try:
                    file_id = img.image_file.file_id
                    outname = f"chart_{file_id}.png"
                    project_client.agents.save_file(file_id=file_id, file_name=outname)
                    print(f"🖼️ Saved image output: {outname}")
                except Exception as img_error:
                    print(f"Note: Could not save image: {img_error}")

    except Exception as e:
        print(f"❌ Error viewing agent responses: {e}")

# Display BMI calculations
if bmi_thread and bmi_run:
    print("\n=== BMI Calculation Results ===")
    if isinstance(bmi_thread, LocalThread):
        view_agent_responses(bmi_thread, is_bmi=True, bmi_value=25.8)
    else:
        view_agent_responses(bmi_thread, thread_id=bmi_thread.id, is_bmi=True)

# Display nutrition analyses
if nutrition_thread and nutrition_run:
    print("\n=== Nutrition Analysis Results ===")
    if isinstance(nutrition_thread, LocalThread):
        if health_agent and hasattr(health_agent, 'file_data'):
            view_agent_responses(nutrition_thread, is_bmi=False, csv_data=health_agent.file_data)
        else:
            view_agent_responses(nutrition_thread, is_bmi=False)
    else:
        view_agent_responses(nutrition_thread, thread_id=nutrition_thread.id, is_bmi=False)

## 6. Cleanup & Best Practices
We can remove our agent and sample data if desired. In production, you might keep them for repeated usage.

### Best Practices in a Nutshell
1. **Data Handling** – Validate input data, handle missing values, properly manage file attachments.
2. **Calculations** – Provide formula steps, disclaimers, limit scope to general wellness, remind user you're not a doctor.
3. **Visualizations** – Use clear labeling and disclaimers that charts are for educational demonstrations.
4. **Security** – Monitor usage, limit access to code interpreter if dealing with proprietary data.


In [ ]:
def cleanup_all():
    """Clean up agent resources and local files.
    
    Handles both server-side agent service resources and local fallback resources.
    """
    try:
        # Delete uploaded file from server if it exists
        if uploaded_file is not None:
            try:
                project_client.agents.delete_file(uploaded_file.id)
                print("🗑️ Deleted uploaded file from agent service.")
            except Exception as e:
                print(f"Note: Could not delete server-side file: {e}")

        # Delete server-side agent if it exists
        if health_agent is not None and not isinstance(health_agent, LocalAgent):
            try:
                project_client.agents.delete_agent(health_agent.id)
                print("🗑️ Deleted health calculator agent.")
            except Exception as e:
                print(f"Note: Could not delete server-side agent: {e}")
        elif isinstance(health_agent, LocalAgent):
            print("🗑️ Cleaned up local health calculator agent (no server-side deletion needed).")

        # Delete local CSV file
        if sample_file and os.path.exists(sample_file):
            os.remove(sample_file)
            print("🗑️ Deleted local sample CSV file.")

    except Exception as e:
        print(f"❌ Error during cleanup: {e}")

cleanup_all()

# Congratulations! 🎉
You now have a **Health Calculator Agent** with the **Code Interpreter** tool that can:
- Perform **BMI calculations** and disclaim that it's not a doctor.
- **Analyze** simple CSV-based nutrition data and produce insights + charts.
- Return images (charts) and text-based insights.

#### Let's proceed to [3-file-search.ipynb](3-file-search.ipynb)

Happy (healthy) coding! 💪
